# Bolt YOLOv8 Segmentation Training — Direct Upload

Google Drive 없이 파일을 직접 업로드해서 학습합니다. 먼저 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

In [ ]:
!pip install -q ultralytics==8.4.86

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

다음 셀에서 `pose_dataset.zip`과 `prepare_dataset.py`를 동시에 선택하세요.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
ZIP_PATH = Path('/content/pose_dataset.zip')
PREPARE_SCRIPT = Path('/content/prepare_dataset.py')
DATASET_DIR = Path('/content/bolt_seg')
RUNS_DIR = Path('/content/runs')

assert ZIP_PATH.is_file(), f'파일 없음: {ZIP_PATH}'
assert PREPARE_SCRIPT.is_file(), f'파일 없음: {PREPARE_SCRIPT}'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('업로드 완료:', ', '.join(uploaded.keys()))

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, str(PREPARE_SCRIPT),
    '--source', str(ZIP_PATH),
    '--output', str(DATASET_DIR),
    '--overwrite',
], check=True)
print((DATASET_DIR / 'data.yaml').read_text())

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n-seg.pt')
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=1024,
    batch=4,
    device=0,
    workers=2,
    weight_decay=0.0005,
    degrees=10,
    translate=0.1,
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    fliplr=0.5,
    close_mosaic=10,
)
print('학습 결과:', results.save_dir)

In [ ]:
best_path = Path(results.save_dir) / 'weights' / 'best.pt'
trained = YOLO(str(best_path))
metrics = trained.val(data=str(DATASET_DIR / 'data.yaml'), device=device)
print('classes:', trained.names)
print('downloading:', best_path)
files.download(str(best_path))

학습이 끝나면 `best.pt` 다운로드가 자동으로 시작됩니다. 코랩 런타임이 종료되면 `/content`의 데이터와 모델이 사라지므로 바로 내려받으세요.